# A.1 給幾個例子能改變回答嗎？

# A：上下文學習與 RAG

前置：第7章prompt與對話。不需要圖片或MoE。把模型想成可以讀你遞給它的小抄的答題者：先手動給例子，再從本地文件找相關段落。檢索成功不等於回答一定正確。




In [ ]:
from pathlib import Path
import os
import subprocess
import sys

# Colab 會先下載專案；本機 Jupyter 沿用已開啟的 repo。
try:
    import google.colab
except ImportError:
    in_colab = False
else:
    in_colab = True

if in_colab:
    root = Path("/content/tiny-perceptron-vlm")
    if not root.exists():
        subprocess.run(
            ["git", "clone", "--depth", "1", "https://github.com/birdhackor/tiny-perceptron-vlm.git", str(root)],
            env={**os.environ, "GIT_LFS_SKIP_SMUDGE": "1"},
            check=True,
        )
    subprocess.run([sys.executable, "-m", "pip", "install", "--quiet", "-e", str(root)], check=True)
    os.chdir(root)

root = Path.cwd().resolve()
while not (root / "pyproject.toml").exists() and root != root.parent:
    root = root.parent
if not (root / "tiny_perceptron").is_dir():
    raise RuntimeError("本機請先依 course/first-steps.md 開啟 repo；免安裝練習可使用本節的 Colab 入口")
sys.path.insert(0, str(root))
import torch
from torch import nn
from torch.nn import functional as F

torch.set_num_threads(1)
torch.manual_seed(42)

**先看圖：像先找小抄，再核對答案來源**

找到對文件與用對資料，是兩個不同分數。

橘色邊框只提示閱讀順序，靜態標註一直保留。系統的減少動態效果設定會停用動畫。


In [ ]:
from IPython.display import SVG, display

display(SVG(filename=str(root / "course/figures/rag.svg")))

**先想一想：**放兩題示例後，關掉對話還會永久保留知識嗎？

zero-shot只給任務，few-shot額外給示例。例子在context裏，不更新權重；必須與真正SFT區分。

**把直覺寫成數學：**同weights，p(answer|prompt_with_examples)可能不同；這是輸入條件變化。

**最小實驗：**以下程式只處理這節的問題。


In [ ]:
zero = "把數字翻倍：3→?"
few = "示例：1→2；2→4。現在：3→?"
print(zero, few, sep="\n")
print("實際比較固定checkpoint與生成參數，不能用規則函數假裝模型學會")

**如何讀結果：**這是prompt實驗設置，不是已訓練模型輸出。真正使用scripts/infer.py --chat 兩次。

**只改一件事：**只換示例順序，保持題目。
